In [ ]:
# # CH-3 核心解释实验：超额收益、因子与中国异象
#
# 本 notebook 是论文复现的主线实证部分，按研究问题组织，而不是按论文表格顺序组织。它接在 `CH3_jup/reproduction_pipeline/01_data_and_factors.ipynb` 后运行，使用该 notebook 已经形成的月度收益、CH-3 因子和异象组合；个股时间序列回归函数直接从 `src_jup/time_series_regression_6.ipynb` 的定义块读取。
#
# 核心问题有三组：CH-3 的规模和价值因子是否改善对中国已报道异象超额收益的解释；FF-5 的 profitability 与 investment 因子在 CH-3 下是否仍有显著定价误差；CH-3 对十项正文异象的多空收益能解释到什么程度。所有显著性判断都保留实际样本期，并把“未拒绝零 alpha”限定为样本证据。
#
# > 本 notebook 的盈利、盈利能力和投资组合仍沿用本地数据代理变量。变量口径和覆盖限制以 01 与 06 notebook 为准。
#
import ast

import copy

import json

from pathlib import Path



import numpy as np

import pandas as pd

import statsmodels.api as sm

from IPython.display import display



project_root = next(path for path in [Path.cwd()] + list(Path.cwd().parents) if (path / 'data' / 'raw').exists())

src_jup_dir = project_root / 'jupter_notebook_version' / 'src_jup'

required_names = ['f_ch3', 'f_ff5', 'anomaly_uncon', 'anomaly_neutral', 'monthly_all', 'mkt', 'smb', 'vmg', 'monthly_excess_ret']

missing_names = [name for name in required_names if name not in globals()]

if missing_names:

    raise RuntimeError('请先按 CH_3_factors.ipynb 的顺序运行 CH3_jup/reproduction_pipeline/01_data_and_factors.ipynb；缺少：{}'.format(missing_names))


In [ ]:
# ## 1. 直接复用 src_jup 的个股时间序列回归
#
# 源函数逐个资产估计超额收益对因子的暴露和截距，使用 White HC1 标准误。本 notebook 只提取函数定义，不触发源 notebook 中读取大表的演示代码。下面把十个论文正文异象的多空收益作为待解释资产。
#
def load_source_function(notebook_name, cell_index, function_name, alias):

    notebook_path = src_jup_dir / notebook_name

    notebook = json.loads(notebook_path.read_text(encoding='utf-8'))

    source = ''.join(notebook['cells'][cell_index]['source'])

    tree = ast.parse(source)

    node = next(

        item for item in tree.body

        if isinstance(item, (ast.FunctionDef, ast.AsyncFunctionDef)) and item.name == function_name

    )

    node = copy.deepcopy(node)

    node.name = alias

    namespace = globals()

    exec(compile(ast.Module(body=[node], type_ignores=[]), str(notebook_path), 'exec'), namespace)

    return namespace[alias]



source_time_series_regression = load_source_function(

    'time_series_regression_6.ipynb', 2, 'time_series_regression', 'source_time_series_regression'

)

print('直接复用：src_jup/time_series_regression_6.ipynb → time_series_regression')


In [ ]:
# ## 2. 规模与价值因子解释中国已报道异象
#
# 用正文十项多空收益作为资产，分别估计 CAPM 和 CH-3。除 alpha 与 t 值外，同一共同月份上报告 R² 的变化，便于看出加入规模、价值因子后模型解释力如何变化。组合收益、因子收益均为月度超额收益。
#
core_anomaly_names = ['size', 'ep', 'bm', 'cp', 'roe', 'volatility', 'max', 'reversal', 'turnover', 'abnormal_turnover']

core_returns = anomaly_uncon.reindex(columns=core_anomaly_names).copy()

capm_factors = f_ch3[[mkt]].copy()

ch3_factors = f_ch3[[mkt, smb, vmg]].copy()



capm_beta, capm_alpha, capm_alpha_t = source_time_series_regression(core_returns, capm_factors)

ch3_beta, ch3_alpha, ch3_alpha_t = source_time_series_regression(core_returns, ch3_factors)



core_rows = []

for anomaly_name in core_anomaly_names:

    aligned = pd.concat([core_returns[anomaly_name].rename(monthly_excess_ret), capm_factors, ch3_factors[[smb, vmg]]], axis=1)

    aligned = aligned.replace([np.inf, -np.inf], np.nan).dropna()

    if aligned.empty:

        continue

    capm_x = sm.add_constant(aligned[[mkt]], has_constant='add')

    ch3_x = sm.add_constant(aligned[[mkt, smb, vmg]], has_constant='add')

    capm_fit = sm.OLS(aligned[monthly_excess_ret], capm_x).fit()

    ch3_fit = sm.OLS(aligned[monthly_excess_ret], ch3_x).fit()

    ch3_white_fit = sm.OLS(aligned[monthly_excess_ret], ch3_x).fit(cov_type='HC1')

    core_rows.append({

        'anomaly': anomaly_name,

        'mean_return_pct': 100 * aligned[monthly_excess_ret].mean(),

        'capm_alpha_pct': 100 * capm_alpha.get(anomaly_name, np.nan),

        'capm_t_alpha': capm_alpha_t.get(anomaly_name, np.nan),

        'capm_r2': capm_fit.rsquared,

        'ch3_alpha_pct': 100 * ch3_alpha.get(anomaly_name, np.nan),

        'ch3_t_alpha': ch3_alpha_t.get(anomaly_name, np.nan),

        'ch3_beta_smb': ch3_beta.loc[smb, anomaly_name] if anomaly_name in ch3_beta else np.nan,

        'ch3_beta_vmg': ch3_beta.loc[vmg, anomaly_name] if anomaly_name in ch3_beta else np.nan,

        'ch3_r2': ch3_fit.rsquared,

        'delta_r2': ch3_fit.rsquared - capm_fit.rsquared,

        'months': len(aligned),

    })

core_anomaly_explanation = pd.DataFrame(core_rows).set_index('anomaly')

display(core_anomaly_explanation)


In [ ]:
core_model_summary = pd.DataFrame([
    {
        'model': model_name,
        'anomalies': int(table[alpha_column].notna().sum()),
        'alpha_significant_5pct': int((table[t_column].abs() >= 1.96).sum()),
        'alpha_significant_1pct': int((table[t_column].abs() >= 2.576).sum()),
        'mean_abs_alpha_pct': table[alpha_column].abs().mean(),
        'mean_r2': table[r2_column].mean(),
        'months_min': int(table['months'].min()),
        'months_max': int(table['months'].max()),
    }
    for model_name, alpha_column, t_column, r2_column in [
        ('CAPM', 'capm_alpha_pct', 'capm_t_alpha', 'capm_r2'),
        ('CH-3', 'ch3_alpha_pct', 'ch3_t_alpha', 'ch3_r2'),
    ]
    for table in [core_anomaly_explanation]
])
display(core_model_summary)
print('判读边界：显著 alpha 数减少或 R² 上升是本地样本的解释力证据；不能把未拒绝零 alpha 写成已证明 alpha 等于零。')


In [ ]:
# ## 3. Profitability 与 investment 因子是否还留下定价误差
#
# 将本地构造的 FF-5 profitability 和 investment 因子收益分别作为待解释收益，对 CH-3 的市场、规模、价值因子做时间序列回归。`ff_profit_factor` 使用 ROE 代理，`ff_invest_factor` 使用资产增长代理；两者财务样本明显短于主因子样本，结果只回答当前代理和有效月份下是否仍能拒绝零 alpha。
#
required_ff5_columns = ['ff_profit_factor', 'ff_invest_factor']

if not set(required_ff5_columns).issubset(f_ff5.columns):

    raise KeyError('f_ff5 缺少列：{}'.format(sorted(set(required_ff5_columns) - set(f_ff5.columns))))



ff5_target_returns = f_ff5[required_ff5_columns].copy()

ff5_target_beta, ff5_target_alpha, ff5_target_alpha_t = source_time_series_regression(ff5_target_returns, ch3_factors)



factor_explanation_rows = []

for factor_name in required_ff5_columns:

    aligned = pd.concat([ff5_target_returns[factor_name].rename('asset_return'), ch3_factors], axis=1)

    aligned = aligned.replace([np.inf, -np.inf], np.nan).dropna()

    t_value = ff5_target_alpha_t.get(factor_name, np.nan)

    factor_explanation_rows.append({

        'target_factor': factor_name,

        'ch3_alpha_pct': 100 * ff5_target_alpha.get(factor_name, np.nan),

        't_alpha': t_value,

        'months': len(aligned),

        'conclusion_at_5pct': '未拒绝 alpha=0' if np.isfinite(t_value) and abs(t_value) < 1.96 else '仍拒绝 alpha=0',

    })

ch3_explains_ff5_proxies = pd.DataFrame(factor_explanation_rows).set_index('target_factor')

display(ch3_explains_ff5_proxies)


In [ ]:
# ## 4. CH-3 因子收益与研究主线结论
#
# 因子均值表用于确认规模和价值风险溢价在本地样本中的方向与统计量。主线结论来自上面的异象多空组合 alpha/R² 和 profitability/investment 代理因子 alpha；与 CH-3 对 FF-3 因子的互相定价及 Gibbons 检验、EP 代理变量赛马、壳价值 30% 依据、CH-4 加入换手因子后的定价检验，放在 `src_jup/CH3_补充定价检验.ipynb` 中展开。
#
core_factor_summary = pd.DataFrame({

    'mean_pct': 100 * ch3_factors.mean(),

    'std_pct': 100 * ch3_factors.std(ddof=1),

    't_mean': ch3_factors.mean() / (ch3_factors.std(ddof=1) / np.sqrt(ch3_factors.count())),

    'months': ch3_factors.count(),

})

display(core_factor_summary)



core_conclusion_snapshot = pd.DataFrame([

    {'question': 'CH-3 相对 CAPM 对十项正文异象的平均解释力', 'evidence': '平均绝对 alpha、显著 alpha 数和共同样本 R²，见 core_model_summary'},

    {'question': '盈利能力代理因子在 CH-3 下是否仍有 alpha', 'evidence': ch3_explains_ff5_proxies.loc['ff_profit_factor', 'conclusion_at_5pct']},

    {'question': '投资代理因子在 CH-3 下是否仍有 alpha', 'evidence': ch3_explains_ff5_proxies.loc['ff_invest_factor', 'conclusion_at_5pct']},

])

display(core_conclusion_snapshot)
